## Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.metrics import roc_curve
import tqdm
import glob

## Metric Functions

In [ ]:
def calculate_eer(y_true, y_pred_proba):
    """Calculates the Equal Error Rate (EER)."""
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    fnr = 1 - tpr
    eer_threshold = thresholds[np.nanargmin(np.absolute((fnr - fpr)))]
    eer = fpr[np.nanargmin(np.absolute((fnr - fpr)))]
    return eer, eer_threshold


def calculate_metrics(y_true, y_pred_proba, y_true_spoof_type, real_label, y_pred_std, use_std=False):
    """Calculates several metrics."""

    # AUC
    auc = roc_auc_score(y_true, y_pred_proba)

    # EER and EER threshold
    eer, eer_threshold = calculate_eer(y_true, y_pred_proba)

    y_pred = (y_pred_proba >= eer_threshold).astype(int)  # Assuming eer_threshold as threshold for classification

    std_min_miss = np.mean(y_pred_std[np.where(y_true!=y_pred)[0]])
    std_max_assert = np.mean(y_pred_std[np.where(y_pred==y_pred)[0]])

    std_diff_rel = (std_min_miss-std_max_assert)/(std_max_assert+1e-15)
    if use_std and std_diff_rel>3:
        std_threshold = min(np.abs(std_min_miss + std_max_assert)/2*1.5, std_min_miss)
        y_pred_std_bool = (y_pred_std <= std_threshold).astype(int)
        y_pred = y_pred*y_pred_std_bool
        y_pred_proba = y_pred_proba*y_pred_std_bool

    # Accuracy
    accuracy = accuracy_score(y_true, y_pred)

    # FPR, FNR, HTER at EER threshold
    fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
    fnr = 1 - tpr

    eer_index = np.nanargmin(np.absolute((fnr - fpr)))
    far_eer = fpr[eer_index]
    frr_eer = fnr[eer_index]
    hter_eer = (far_eer + frr_eer) / 2
    BPCER = frr_eer

    APCER=0
    for i in np.unique(y_true_spoof_type):
        if i!=real_label:
            y_true_spoof = y_true[np.where(y_true_spoof_type==i)[0]]
            y_pred_proba_spoof = y_pred_proba[np.where(y_true_spoof_type==i)[0]]

            y_pred_spoof = (y_pred_proba_spoof >= eer_threshold).astype(int)

            if use_std and std_diff_rel>3:
                y_pred_std_spoof = y_pred_std[np.where(y_true_spoof_type==i)[0]]
                y_pred_std_bool_spoof = (y_pred_std_spoof <= std_threshold).astype(int)
                y_pred_spoof = y_pred_spoof*y_pred_std_bool_spoof


            conf_matrix = confusion_matrix(y_true_spoof, y_pred_spoof, labels=[0,1])

            tn_fp_fn_tp = conf_matrix.flatten()
            if len(tn_fp_fn_tp)==4:
                tn, fp, fn, tp = tn_fp_fn_tp
            elif len(tn_fp_fn_tp)==3:
                tn, fp, fn= tn_fp_fn_tp
                tp = 0
            elif len(tn_fp_fn_tp)==2:
                tn, fp= tn_fp_fn_tp
                tp = 0
                fn = 0
            elif len(tn_fp_fn_tp)==1:
                tn= tn_fp_fn_tp
                tp = 0
                fn = 0
                fp = 0
            else:
                tn, fp, fn, tp = [0,0,0,0]

            far_spoof = min(1.0, fp / (fp + tn + 0.00000001))
            # print('spoof:',i,'apcer:',far_spoof)

            APCER=max(far_spoof,APCER)

    ACER=(APCER+BPCER) / 2

    return auc, accuracy, eer, hter_eer, eer_threshold, APCER, BPCER, ACER, conf_matrix, std_min_miss, std_max_assert, std_diff_rel


def evaluate_data(model_reference, path_csv, model_probabilistic, use_std=False):

    df_test = pd.read_csv(path_csv, index_col=False)
    if 'aenet' in model_reference.lower():
        df_test['liveness_final_label'] = df_test.label.apply(lambda x: ([0, 1] if x.lower() in ['real', 'live'] else [1, 0]) if str(x)!='nan' else np.nan)
        df_test['liveness_final_pred'] = df_test.label_predict_proba.apply(lambda x: eval(str(x)) if str(x)!='nan' else np.nan)
    else:
        df_test.liveness_final_label = df_test.liveness_final_label.apply(lambda x: str(x).replace(' ', ', ')).apply(lambda x: eval(x) if str(x)!='nan' else np.nan)
        df_test.liveness_final_pred = df_test.liveness_final_pred.apply(lambda x: eval(str(x)) if str(x)!='nan' else np.nan)

    if 'aenet' in model_reference.lower():
        df_test.liveness_final_pred = df_test.liveness_final_pred.apply(lambda x: [x[1], x[0]])

    df_test = df_test[~df_test.liveness_final_pred.isnull()].reset_index(drop=True)
    y_true = np.array(df_test.liveness_final_label.tolist())[:, 1]
    if model_probabilistic:
        y_pred_proba = np.array([np.mean(x) for x in df_test.liveness_final_pred])
        y_pred_std = np.array([np.std(x) for x in df_test.liveness_final_pred])
    else:
        y_pred_proba = np.array([x[1] for x in df_test.liveness_final_pred])
        y_pred_std = np.zeros(len(y_pred_proba))

    if 'scene' in df_test:
        spoof_type = df_test.scene.apply(lambda x: x.split('_')[-2])
        spoof_type_dict = dict(zip(spoof_type.unique(), range(spoof_type.nunique())))
        real_label = spoof_type_dict['SD']
        y_true_spoof_type = np.array(spoof_type.map(spoof_type_dict).tolist())
    else:
        real_label = 0
        y_true_spoof_type = np.array(df_test.spoof_type.tolist())

    auc, accuracy, eer, hter_eer, eer_threshold, APCER, BPCER, ACER, conf_matrix, std_min_miss, std_max_assert, std_diff_rel = calculate_metrics(y_true, y_pred_proba, y_true_spoof_type, real_label, y_pred_std, use_std=use_std)

    # print(f"AUC: {auc}")
    # print(f"Accuracy: {accuracy}")
    # print(f"EER: {eer}")
    # print(f"EER Threshold: {eer_threshold}")
    # print(f"FAR at EER: {far_eer}")
    # print(f"FRR at EER: {frr_eer}")
    # print(f"HTER at EER: {hter_eer}")
    # print(f"Confusion Matrix:\n{conf_matrix}")

    return {
            'Model': model_reference,
            'EER': eer,
            'HTER': hter_eer,
            'AUC': auc,
            'APCER': APCER,
            'BPCER': BPCER,
            'ACER': ACER,
            'Threshold': eer_threshold,
            'ACC': accuracy,
            'STD-error':std_min_miss,
            'STD-assert': std_max_assert,
            'STD-diff-rel': std_diff_rel

        }

## Load Result Data

In [ ]:
common_path = '/content/gdrive/MyDrive/csv_results/'

In [ ]:
model_dict_CEAS_min = {
    'PhD1_protIntra_trCeAS_valCeAS_min': (f'{common_path}PhD1_protIntra_trCeAS_valCeAS_min.csv', False),
    'PhD2_protIntra_trCeAS_valCeAS_min': (f'{common_path}PhD2_protIntra_trCeAS_valCeAS_min.csv', True),
    'PhD3_protIntra_trCeAS_valCeAS_min': (f'{common_path}PhD3_protIntra_trCeAS_valCeAS_min.csv', False),
    'PhD4_protIntra_trCeAS_valCeAS_min': (f'{common_path}PhD4_protIntra_trCeAS_valCeAS_min.csv', True),
    'PhD5_protIntra_trCeAS_valCeAS_min': (f'{common_path}PhD5_protIntra_trCeAS_valCeAS_min.csv', True),
}

model_dict_CEAS_protocols = {
    'PhD4_protIntraFull_trCeAS_tsCeAS': (f'{common_path}PhD4_protIntraFull_trCeAS_tsCeAS.csv', True),
    'PhD4_prot1_trCeAS_tsCeAS': (f'{common_path}PhD4_prot1_trCeAS_tsCeAS.csv', True),
    'AENet_prot3_trCeAS_tsMSU': (f'{common_path}AENet_prot3_trCeAS_tsMSU.csv', False),
    'PhD4_protCrossDtSt_trCeAS_tsMSU': (f'{common_path}PhD4_protCrossDtSt_trCeAS_tsMSU.csv', True),
    'PhD4_protCrossDtSt_trMSU_tsCeAS': (f'{common_path}PhD4_protCrossDtSt_trMSU_tsCeAS.csv', True),
}

model_dict_MSU_min = {
    'PhD1_protIntra_trMSU_valMSU_min': (f'{common_path}PhD1_protIntra_trMSU_valMSU_min.csv', False),
    'PhD2_protIntra_trMSU_valMSU_min': (f'{common_path}PhD2_protIntra_trMSU_valMSU_min.csv', True),
    'PhD3_protIntra_trMSU_valMSU_min': (f'{common_path}PhD3_protIntra_trMSU_valMSU_min.csv', False),
    'PhD4_protIntra_trMSU_valMSU_min': (f'{common_path}PhD4_protIntra_trMSU_valMSU_min.csv', True),
    'PhD5_protIntra_trMSU_valMSU_min': (f'{common_path}PhD5_protIntra_trMSU_valMSU_min.csv', True),
}

model_dict_MSU_protocols = {
    'PhD4_protIntra_trMSU_tsMSU': (f'{common_path}PhD4_protIntra_trMSU_valMSU.csv', True),
}

## Calculate Metrics

### Results for selection of Variant Model

In [11]:
model_dict = model_dict_CEAS_min
model_results = []
for i in tqdm.tqdm(model_dict.keys(), total=len(model_dict)):
    model_reference = i
    path_csv = model_dict[model_reference][0]
    model_probabilistic = model_dict[model_reference][1]
    try:

        model_results.append(evaluate_data(model_reference, path_csv, model_probabilistic, use_std=True))
    except:
        print('Not finded', i)

df_model_CEAS_min = pd.DataFrame(model_results)
df_model_CEAS_min

100%|██████████| 5/5 [00:17<00:00,  3.44s/it]


,Model,EER,HTER,AUC,APCER,BPCER,ACER,Threshold,ACC,STD-error,STD-assert,STD-diff-rel
,,,,,,,,,,,,
0,PhD1_protIntra_trCeAS_valCeAS_min,0.061061,0.060591,0.987113,0.150000,0.060120,0.105060,0.142914,0.939252,0.000000,0.000000,0.000000
1,PhD2_protIntra_trCeAS_valCeAS_min,0.055055,0.129699,0.990114,0.020833,0.138277,0.079555,0.205738,0.938585,0.102769,0.020203,4.086905
2,PhD3_protIntra_trCeAS_valCeAS_min,0.050050,0.051077,0.986782,0.084112,0.052104,0.068108,0.243951,0.949266,0.000000,0.000000,0.000000
3,PhD4_protIntra_trCeAS_valCeAS_min,0.054054,0.119677,0.986504,0.011364,0.114228,0.062796,0.208257,0.951936,0.114011,0.020205,4.642827
4,PhD5_protIntra_trCeAS_valCeAS_min,0.104104,0.104156,0.952516,0.203252,0.104208,0.153730,0.262146,0.895861,0.064508,0.030066,1.145577


In [ ]:
model_dict = model_dict_MSU_min
model_results = []
for i in tqdm.tqdm(model_dict.keys(), total=len(model_dict)):
    model_reference = i
    path_csv = model_dict[model_reference][0]
    model_probabilistic = model_dict[model_reference][1]
    try:

        model_results.append(evaluate_data(model_reference, path_csv, model_probabilistic, use_std=True))
    except:
        print('Not finded', i)

df_model_MSU_min = pd.DataFrame(model_results)

df_model_MSU_min

100%|██████████| 5/5 [00:01<00:00,  3.29it/s]


,Model,EER,HTER,AUC,APCER,BPCER,ACER,Threshold,ACC,STD-error,STD-assert,STD-diff-rel
0,PhD1_protIntra_trMSU_valMSU_min,0.043011,0.043641,0.980310,0.051768,0.044271,0.048019,0.053791,0.956667,0.000000,0.000000,0.000000
1,PhD2_protIntra_trMSU_valMSU_min,0.060932,0.179449,0.975179,0.054293,0.179688,0.116990,0.009985,0.914000,0.030244,0.006912,3.375490
2,PhD3_protIntra_trMSU_valMSU_min,0.062724,0.063914,0.984569,0.075758,0.065104,0.070431,0.200162,0.936667,0.000000,0.000000,0.000000
3,PhD4_protIntra_trMSU_valMSU_min,0.060932,0.060414,0.977853,0.061728,0.059896,0.060812,0.011875,0.939333,0.051517,0.015297,2.367739
4,PhD5_protIntra_trMSU_valMSU_min,0.057348,0.204203,0.982149,0.049383,0.200521,0.124952,0.025990,0.914667,0.060466,0.012218,3.948849


### Protocols

In [10]:
model_dict = model_dict_CEAS_protocols
model_results = []
for i in tqdm.tqdm(model_dict.keys(), total=len(model_dict)):
    model_reference = i
    path_csv = model_dict[model_reference][0]
    model_probabilistic = model_dict[model_reference][1]
    try:

        model_results.append(evaluate_data(model_reference, path_csv, model_probabilistic, use_std=True))
    except:
        print('Not finded', i)

df_model_CEAS_protocols = pd.DataFrame(model_results)
df_model_CEAS_protocols

100%|██████████| 5/5 [00:17<00:00,  3.45s/it]


,Model,EER,HTER,AUC,APCER,BPCER,ACER,Threshold,ACC,STD-error,STD-assert,STD-diff-rel
,,,,,,,,,,,,
0,PhD4_protIntraFull_trCeAS_tsCeAS,0.053294,0.053276,0.987853,0.119678,0.053258,0.086468,0.644493,0.946716,0.090651,0.027457,2.301574
1,PhD4_prot1_trCeAS_tsCeAS,0.059002,0.059018,0.985106,0.122279,0.059035,0.090657,0.922221,0.940987,0.060614,0.040264,0.505410
2,AENet_prot3_trCeAS_tsMSU,0.320554,0.320524,0.754256,0.453081,0.320494,0.386787,0.932475,0.679460,0.000000,0.000000,0.000000
3,PhD4_protCrossDtSt_trCeAS_tsMSU,0.439785,0.439788,0.600987,0.605556,0.439792,0.522674,0.972542,0.560213,0.029380,0.047079,-0.375942
4,PhD4_protCrossDtSt_trMSU_tsCeAS,0.457701,0.457643,0.547378,0.648757,0.457585,0.553171,0.005628,0.542334,0.046830,0.033207,0.410275


In [15]:
model_dict = model_dict_MSU_protocols
model_results = []
for i in tqdm.tqdm(model_dict.keys(), total=len(model_dict)):
    model_reference = i
    path_csv = model_dict[model_reference][0]
    model_probabilistic = model_dict[model_reference][1]
    try:

        model_results.append(evaluate_data(model_reference, path_csv, model_probabilistic, use_std=True))
    except:
        print('Not finded', i)

df_model_MSU_protocols = pd.DataFrame(model_results)
df_model_MSU_protocols

100%|██████████| 1/1 [00:02<00:00,  2.59s/it]


,Model,EER,HTER,AUC,APCER,BPCER,ACER,Threshold,ACC,STD-error,STD-assert,STD-diff-rel
,,,,,,,,,,,,
0,PhD4_protIntra_trMSU_tsMSU,0.047240,0.047058,0.978590,0.030303,0.137917,0.084110,0.096104,0.932960,0.064546,0.009518,5.781826


END